In [17]:
import torch
import torch.nn.functional as F

def compute_bathtub_patch_vectorized(u_coarse_patch, topo_patch, f, max_iters=20):
    """
    Vectorized bathtub (hydrostatic fill) for local patches or full domains.
    Handles rectangular grids (ny != nx).
    """
    # Extract dimensions explicitly
    # Expected: [BS, Height, Width, Time]
    bs, n_y, n_x, n_t = u_coarse_patch.shape
    nf_y, nf_x = n_y * f, n_x * f
    device = u_coarse_patch.device

    # 1. Reshape topo into patches matching coarse cells: [BS, ny, f, nx, f]
    # This must match topo_patch.shape [bs, ny*f, nx*f]
    topo_folded = topo_patch.view(bs, n_y, f, n_x, f).permute(0, 1, 3, 2, 4)
    
    # 2. Flatten fine spatial dims per cell for bisection: [BS, ny, nx, f*f]
    z_fine = topo_folded.reshape(bs, n_y, n_x, f*f)
    
    # 3. Add time dimension for broadcasting: [BS, ny, nx, 1, f*f]
    z_fine = z_fine.unsqueeze(3) 
    # Target coarse depth: [BS, ny, nx, n_t, 1]
    d_target = u_coarse_patch.unsqueeze(-1)

    # 4. Initialize Bisection Bounds
    h_low = z_fine.min(dim=-1, keepdim=True)[0]
    h_high = z_fine.max(dim=-1, keepdim=True)[0] + d_target + 1e-3

    # 5. Vectorized Bisection Loop
    for _ in range(max_iters):
        h_mid = (h_low + h_high) / 2.0
        # Average depth at h_mid: mean( max(H - z, 0) )
        d_mid = torch.relu(h_mid - z_fine).mean(dim=-1, keepdim=True)
        
        mask = d_mid < d_target
        h_low = torch.where(mask, h_mid, h_low)
        h_high = torch.where(mask, h_high, h_mid)

    # 6. Final Reconstruction
    h_final = (h_low + h_high) / 2.0
    u_rec = torch.relu(h_final - z_fine) # [BS, ny, nx, n_t, f*f]

    # 7. Reconstruct spatial grid shape: [BS, ny*f, nx*f, n_t]
    # View: [BS, ny, nx, n_t, f, f] 
    u_rec = u_rec.view(bs, n_y, n_x, n_t, f, f)
    # Permute to put ny and f together, nx and f together: [BS, ny, f, nx, f, n_t]
    u_rec = u_rec.permute(0, 1, 4, 2, 5, 3).reshape(bs, nf_y, nf_x, n_t)

    return u_rec

In [24]:
def test_bathtub():
    # Configuration
    bs = 2
    f = 4
    
    # Standardize: my is height (Rows), mx is width (Cols)
    my, mx = 41, 82  
    n_t = 88
    
    # Fine dimensions
    ny = my * f
    nx = mx * f

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Create dummy inputs [Batch, Height, Width, Time]
    u_coarse = torch.rand(bs, my, mx, n_t).to(device) + 0.1
    
    # 2. Topo must match the fine grid: [Batch, Height, Width]
    topo_fine = torch.randn(bs, ny, nx).to(device)

    print(f"Inputs: Coarse {u_coarse.shape}, Topo {topo_fine.shape}")

    # Run function
    # Ensure compute_bathtub_patch_vectorized uses the logic:
    # bs, n_y, n_x, n_t = u_coarse_patch.shape
    u_bathtub = compute_bathtub_patch_vectorized(u_coarse, topo_fine, f)

    print(f"Output: Bathtub HR {u_bathtub.shape}")

    # CHECK 1: Shape Match
    assert u_bathtub.shape == (bs, ny, nx, n_t), f"Shape mismatch! Got {u_bathtub.shape}"

    # CHECK 2: Mass Conservation (The Mean Constraint)
    # Move Time to Channel dim for avg_pool2d: [BS, T, H_fine, W_fine]
    u_bt_permuted = u_bathtub.permute(0, 3, 1, 2) 
    
    # average pooling reduces fine grid back to coarse resolution
    u_re_coarsened = F.avg_pool2d(u_bt_permuted, kernel_size=f).permute(0, 2, 3, 1) # [BS, H_coarse, W_coarse, T]
    
    diff = torch.abs(u_re_coarsened - u_coarse).mean()
    print(f"Mean Difference (Mass Balance): {diff.item():.2e}")
    
    if diff < 1e-4:
        print("Test Passed: Reconstructed fine depth matches coarse mean constraint!")
    else:
        print("Test Failed: Mass balance not preserved.")

if __name__ == "__main__":
    test_bathtub()

Inputs: Coarse torch.Size([2, 41, 82, 88]), Topo torch.Size([2, 164, 328])
Output: Bathtub HR torch.Size([2, 164, 328, 88])
Mean Difference (Mass Balance): 6.18e-07
Test Passed: Reconstructed fine depth matches coarse mean constraint!
